# 05 — Tool calling aplicado a ATM

## De lenguaje natural a acciones sobre un sistema

En 04 construimos un RAG que recupera información y responde. Ahora el objetivo es diferente: convertir una petición en llamadas a funciones y comprobar sus resultados.

Caso de estudio: **comprobar que, al seleccionar una aeronave, el panel de información muestra su plan de vuelo correcto**.

No utilizaremos un sistema ATM real: construiremos un simulador mínimo en memoria. No requiere MCP, Robot Framework ni embeddings. Introduciremos MCP después, cuando comprendamos el contrato entre LLM y aplicación.

Al terminar podrás distinguir: descripción de herramienta, llamada propuesta por el LLM, validación, ejecución, resultado, orquestación y veredicto de prueba.

## 0. Preparación

Desde la raíz de `ai-atm`, si aún no tienes el SDK:

```bash
uv add openai
```

Usa la misma configuración que funcionó en los notebooks 03 y 04: `OPENAI_API_KEY` y, si procede, `OPENAI_MODEL`. No guardes la clave en Git. El valor por defecto se mantiene por continuidad (`gpt-6-luna`); si tu cuenta usa otro identificador, configura `OPENAI_MODEL`.

Las celdas marcadas **API** hacen peticiones facturables. Las pruebas locales no necesitan una clave ni red. Solo se envían datos sintéticos; no introduzcas información interna sin autorización.

Referencia: [Function calling — Responses API](https://developers.openai.com/api/docs/guides/function-calling).

In [ ]:
import json
import os
from copy import deepcopy
from dataclasses import dataclass, field
from types import SimpleNamespace
from pprint import pprint

MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

## 1. Nuestro sistema simulado (software convencional)

El sistema tiene un plan de vuelo de referencia y un panel que muestra la aeronave seleccionada. Los valores son ficticios. El panel puede devolver un destino erróneo para simular un defecto.

Observa que estas funciones no contienen IA. Su comportamiento es determinista. El LLM será quien proponga cuándo invocarlas.

In [ ]:
FLIGHT_PLANS = {
    "ABC123": {"callsign": "ABC123", "departure": "LEMD", "destination": "LEBL", "flight_level": 330},
    "XYZ456": {"callsign": "XYZ456", "departure": "LEBL", "destination": "LEPA", "flight_level": 290},
}

@dataclass
class ATMSimulator:
    selected: str | None = None
    faulty_panel: bool = False
    flights: dict = field(default_factory=lambda: deepcopy(FLIGHT_PLANS))

    def get_flight_plan(self, callsign):
        if callsign not in self.flights:
            raise ValueError("Aeronave desconocida")
        return deepcopy(self.flights[callsign])

    def select_aircraft(self, callsign):
        self.get_flight_plan(callsign)  # valida antes de cambiar estado
        self.selected = callsign
        return {"selected": callsign}

    def read_flight_panel(self):
        if self.selected is None:
            raise ValueError("No hay ninguna aeronave seleccionada")
        panel = self.get_flight_plan(self.selected)
        if self.faulty_panel:
            panel["destination"] = "LEVC"  # defecto deliberado
        return panel

sim = ATMSimulator()
pprint(sim.get_flight_plan("ABC123"))
pprint(sim.select_aircraft("ABC123"))
pprint(sim.read_flight_panel())

## 2. Dar a conocer las herramientas al LLM

El modelo no necesita recibir el código Python. Recibe un contrato: nombre, descripción y esquema JSON de argumentos. Las descripciones ayudan a elegir la herramienta adecuada. El esquema delimita la estructura de los argumentos.

En la Responses API, la definición de una función es un elemento con `type: function`. Con `strict=True`, todos los campos definidos deben ser requeridos y los objetos deben prohibir propiedades adicionales. Esto no verifica la existencia de una aeronave ni autoriza una acción: esa responsabilidad sigue siendo nuestra.

In [ ]:
def function_tool(name, description, properties):
    return {
        "type": "function",
        "name": name,
        "description": description,
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": properties,
            "required": list(properties),
            "additionalProperties": False,
        },
    }

CALLSIGN = {"type": "string", "description": "Identificador de la aeronave, por ejemplo ABC123."}
TOOLS = [
    function_tool("get_flight_plan", "Consulta el plan de vuelo de referencia. No modifica el sistema.", {"callsign": CALLSIGN}),
    function_tool("select_aircraft", "Selecciona una aeronave en el simulador de interfaz. Cambia la selección local.", {"callsign": CALLSIGN}),
    function_tool("read_flight_panel", "Lee el panel de la aeronave actualmente seleccionada. Requiere selección previa.", {}),
]

pprint(TOOLS)

## 3. Dispatcher: la frontera entre IA y ejecución

El dispatcher recibe un nombre y argumentos. No usa `eval`, ni ejecuta código arbitrario generado por el modelo. Solo admite funciones conocidas, comprueba argumentos y permite cambios de estado únicamente cuando se autoriza explícitamente en este simulador.

El resultado es un diccionario serializable a JSON. Los errores se devuelven de manera estructurada. En producción habría también autenticación, permisos por recurso, límites, auditoría y aislamiento.

In [ ]:
def dispatch(simulator, name, arguments, *, allow_selection=False):
    allowed = {
        "get_flight_plan": simulator.get_flight_plan,
        "select_aircraft": simulator.select_aircraft,
        "read_flight_panel": simulator.read_flight_panel,
    }
    if name not in allowed:
        return {"ok": False, "error": "Herramienta no permitida"}
    if name == "select_aircraft" and not allow_selection:
        return {"ok": False, "error": "Selección no autorizada"}
    try:
        args = json.loads(arguments)
        expected = set() if name == "read_flight_panel" else {"callsign"}
        if not isinstance(args, dict) or set(args) != expected:
            raise ValueError("Argumentos incorrectos")
        if "callsign" in args:
            if not isinstance(args["callsign"], str) or not args["callsign"].strip():
                raise ValueError("callsign debe ser texto no vacío")
            args["callsign"] = args["callsign"].strip().upper()
        return {"ok": True, "data": allowed[name](**args)}
    except (ValueError, TypeError) as exc:
        return {"ok": False, "error": str(exc)}

sim = ATMSimulator()
pprint(dispatch(sim, "get_flight_plan", '{"callsign":"ABC123"}'))
pprint(dispatch(sim, "select_aircraft", '{"callsign":"ABC123"}'))
pprint(dispatch(sim, "read_flight_panel", '{}'))
assert sim.selected is None

## 4. Primera llamada al modelo (API)

Vamos a pedir el destino de ABC123. La API recibe las herramientas disponibles, pero **no ejecuta nuestras funciones Python**. Inspeccionaremos los elementos de `response.output` y buscaremos `function_call`.

El cliente se crea bajo demanda para que las partes locales puedan ejecutarse sin credenciales. `store=True` permite continuar usando el identificador de la respuesta; las instrucciones se reenvían cada vez. Solo se utilizan datos sintéticos.

In [ ]:
INSTRUCTIONS = """
Eres un asistente para un simulador ATM de pruebas, no para operaciones reales.
Usa las herramientas para obtener datos del simulador; no inventes resultados.
Si una herramienta falla, no afirmes que tuvo éxito.
Para una prueba del panel, consulta el plan de vuelo de referencia, selecciona la aeronave
si es necesario y lee el panel después de seleccionarla. Compara los campos solicitados.
No declares PASS si falta evidencia o si una acción necesaria falló.
No selecciones otra aeronave distinta de la indicada por el usuario.
Responde en español. Si falta un identificador necesario, solicita aclaración.
"""

def get_client():
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Configura OPENAI_API_KEY antes de las celdas API")
    return OpenAI()

def request_model(client, user_input, previous_response_id=None, tools=None):
    params = dict(model=MODEL, instructions=INSTRUCTIONS, input=user_input,
                  tools=TOOLS if tools is None else tools,
                  parallel_tool_calls=False, store=True)
    if previous_response_id is not None:
        params["previous_response_id"] = previous_response_id
    return client.responses.create(**params)

In [ ]:
# API — primera petición
client = get_client()
response = request_model(client, "¿Cuál es el destino del plan de vuelo de ABC123?",
                         tools=TOOLS[:1])
for item in response.output:
    pprint(item.model_dump(exclude_none=True))

Esperamos observar algo equivalente a:

```json
{"type":"function_call", "name":"get_flight_plan", "arguments":"{\"callsign\":\"ABC123\"}", "call_id":"..."}
```

`arguments` es una cadena JSON y `call_id` permite asociar el resultado con esa petición concreta. No confundas `call_id` con el identificador de la respuesta.

Si el modelo responde directamente sin llamar a una herramienta, eso es una decisión observable (y en este ejemplo una posible deficiencia): no finjas que hubo ejecución.

## 5. Ejecutar la petición y devolver el resultado (API)

Ejecutamos las llamadas recibidas y construimos objetos `function_call_output` con el mismo `call_id`. La siguiente petición al LLM incluye el resultado y el identificador de la respuesta anterior. No hay nuevo entrenamiento.

In [ ]:
def function_calls(response):
    return [item for item in response.output if item.type == "function_call"]

def execute_call(simulator, call, *, allow_selection=False):
    result = dispatch(simulator, call.name, call.arguments, allow_selection=allow_selection)
    output = {"type": "function_call_output", "call_id": call.call_id,
              "output": json.dumps(result, ensure_ascii=False)}
    return result, output

In [ ]:
# API — completar un intercambio manual
manual_sim = ATMSimulator()
calls = function_calls(response)
if not calls:
    print("Sin llamadas propuestas:", response.output_text)
elif response.status != "completed":
    print("Respuesta incompleta: no se ejecuta")
else:
    outputs = []
    for call in calls:
        result, output = execute_call(manual_sim, call)
        pprint(output)
        outputs.append(output)
    continuation = request_model(client, outputs, response.id, tools=TOOLS[:1])
    print("Respuesta:", continuation.output_text)
    if function_calls(continuation):
        print("El modelo solicita otra llamada: se gestionará en el bucle siguiente.")

## 6. Bucle de herramientas: orquestación

Para una tarea con varios pasos, una sola llamada no basta. El modelo puede solicitar otra herramienta tras recibir un resultado. Nuestra aplicación controla el bucle, el presupuesto de rondas y la autorización de los cambios de estado.

No es necesario introducir todavía un framework de agentes. Este bucle explícito hace visible qué ocurre. `parallel_tool_calls=False` facilita las dependencias secuenciales (seleccionar antes de leer). Cada llamada y resultado se registra en `trace`.

In [ ]:
def run_task(client, question, simulator, *, allow_selection=False, max_rounds=6):
    if max_rounds < 1:
        raise ValueError("max_rounds debe ser positivo")
    trace = []
    response = request_model(client, question)
    for round_number in range(1, max_rounds + 1):
        if response.status != "completed":
            return {"status": "incomplete", "answer": "", "trace": trace}
        calls = function_calls(response)
        if not calls:
            status = "completed" if response.output_text else "no_answer"
            return {"status": status, "answer": response.output_text, "trace": trace}
        outputs = []
        for call in calls:
            result, output = execute_call(simulator, call, allow_selection=allow_selection)
            trace.append({"round": round_number, "tool": call.name,
                          "arguments": call.arguments, "result": result})
            outputs.append(output)
        response = request_model(client, outputs, response.id)
    if response.status == "completed" and not function_calls(response) and response.output_text:
        return {"status": "completed", "answer": response.output_text, "trace": trace}
    return {"status": "incomplete" if response.status != "completed" else "limit_reached",
            "answer": "", "trace": trace}


def show_task(task):
    for event in task["trace"]:
        print(f"Ronda {event['round']}: {event['tool']}({event['arguments']})")
        pprint(event["result"])
    print("Estado:", task["status"])
    print("Respuesta del LLM:", task["answer"])

## 7. El veredicto de una prueba NO lo decide el LLM

Aquí hacemos un cambio importante respecto a RAG. En una prueba automática, la aplicación debe comparar el resultado observado con el esperado. Una frase del modelo diciendo PASS no demuestra que la prueba pasó.

Para este caso fijo, exigimos evidencia de: plan de referencia de la aeronave pedida, selección exitosa y lectura posterior del panel. Un fallo de ejecución o una evidencia insuficiente produce ERROR/INCONCLUSIVE. Una diferencia entre los datos observados y esperados produce FAIL.

Este verificador pertenece al caso concreto; no es un evaluador universal de pruebas escritas en lenguaje natural.

In [ ]:
def verify_panel_test(task, callsign):
    expected = observed = None
    selected = False
    if task["status"] != "completed":
        return {"verdict": "INCONCLUSIVE", "reason": "Orquestación incompleta"}
    for event in task["trace"]:
        result = event["result"]
        if not result["ok"]:
            return {"verdict": "ERROR", "reason": result["error"]}
        data = result["data"]
        if event["tool"] == "get_flight_plan" and data.get("callsign") == callsign:
            expected = data
        elif event["tool"] == "select_aircraft":
            if data.get("selected") != callsign:
                return {"verdict": "ERROR", "reason": "Aeronave distinta de la solicitada"}
            selected, observed = True, None
        elif event["tool"] == "read_flight_panel" and selected:
            observed = data
    if expected is None or not selected or observed is None:
        return {"verdict": "INCONCLUSIVE", "reason": "Falta evidencia requerida"}
    fields = ("callsign", "departure", "destination", "flight_level")
    if any(key not in expected or key not in observed for key in fields):
        return {"verdict": "INCONCLUSIVE", "reason": "Datos incompletos"}
    differences = {key: {"expected": expected[key], "observed": observed[key]}
                   for key in fields if expected[key] != observed[key]}
    return {"verdict": "FAIL" if differences else "PASS", "differences": differences}

## 8. Pruebas locales reproducibles (sin API)

Para separar la lógica software del comportamiento no determinista del modelo, simulamos las respuestas del proveedor. Este mock no demuestra que un LLM real seleccione correctamente las herramientas. Sí permite verificar el dispatcher, el bucle, la correlación de llamadas y los veredictos.

In [ ]:
class ScriptedResponses:
    def __init__(self, script):
        self.script = list(script)
        self.requests = []
    def create(self, **kwargs):
        self.requests.append(kwargs)
        index = len(self.requests)
        if index > len(self.script):
            raise AssertionError("El bucle hizo una petición inesperada")
        item = self.script[index - 1]
        if isinstance(item, tuple):
            name, arguments = item
            output = [SimpleNamespace(type="function_call", name=name,
                        arguments=json.dumps(arguments), call_id=f"call_{index}")]
            text = ""
        else:
            output, text = [], item
        return SimpleNamespace(id=f"response_{index}", output=output,
                               output_text=text, status="completed")

def scripted_client(script):
    return SimpleNamespace(responses=ScriptedResponses(script))

script = [
    ("get_flight_plan", {"callsign": "ABC123"}),
    ("select_aircraft", {"callsign": "ABC123"}),
    ("read_flight_panel", {}),
    "La prueba ha pasado (texto simulado deliberadamente).",
]

for faulty, expected_verdict in [(False, "PASS"), (True, "FAIL")]:
    fake = scripted_client(script)
    task = run_task(fake, "Comprueba ABC123", ATMSimulator(faulty_panel=faulty),
                    allow_selection=True)
    verdict = verify_panel_test(task, "ABC123")
    assert verdict["verdict"] == expected_verdict, verdict
    assert fake.responses.requests[1]["previous_response_id"] == "response_1"
    assert fake.responses.requests[1]["input"][0]["call_id"] == "call_1"
    print("Simulador con defecto:", faulty, "→", verdict)

assert dispatch(ATMSimulator(), "get_flight_plan", '{"callsign":"MISSING"}')["ok"] is False
assert dispatch(ATMSimulator(), "unknown", '{}')["ok"] is False
assert dispatch(ATMSimulator(), "get_flight_plan", '{"callsign":123}')["ok"] is False
assert dispatch(ATMSimulator(), "get_flight_plan", '{"callsign":"ABC123","extra":1}')["ok"] is False
assert dispatch(ATMSimulator(), "get_flight_plan", 'not json')["ok"] is False
blocked = run_task(scripted_client(script), "Comprueba ABC123", ATMSimulator())
assert verify_panel_test(blocked, "ABC123")["verdict"] == "ERROR"
missing = run_task(scripted_client(["PASS"]), "Comprueba ABC123", ATMSimulator())
assert verify_panel_test(missing, "ABC123")["verdict"] == "INCONCLUSIVE"
limited = run_task(scripted_client(script), "Comprueba ABC123", ATMSimulator(), max_rounds=1)
assert limited["status"] == "limit_reached"
print("Pruebas locales OK")

## 9. Prueba completa con el LLM (API)

Ahora sustituimos el mock por el modelo. Usamos un simulador nuevo por ejecución, sin acceso a sistemas externos. El mismo texto de prueba se ejecuta primero sobre un sistema correcto y luego sobre uno con defecto. Analiza si el modelo propone la secuencia adecuada y si su conclusión coincide con el veredicto determinista.

In [ ]:
# API — escenario normal y con fallo deliberado
client = get_client()
TEST_REQUEST = """
Comprueba que al seleccionar la aeronave ABC123, el panel muestra su plan de vuelo correcto:
identificador, origen, destino y nivel de vuelo, comparados con el plan de referencia.
Indica si hay diferencias y cuáles son.
"""

for faulty in [False, True]:
    print("\n=== Defecto simulado:", faulty, "===")
    task = run_task(client, TEST_REQUEST, ATMSimulator(faulty_panel=faulty), allow_selection=True)
    show_task(task)
    print("Veredicto calculado por Python:")
    pprint(verify_panel_test(task, "ABC123"))

## 10. Experimentos opcionales (API)

Prueba una consulta sin acción, una aeronave inexistente y una petición sin identificador. Observa la traza. Por último intenta seleccionar sin autorización. El modelo puede proponer la acción, pero el dispatcher debe rechazarla.

Cada caso comienza con estado limpio. No se usan herramientas reales ni se realizan cambios persistentes.

In [ ]:
# API — opcional, descomenta un caso cada vez para controlar el coste
examples = [
    ("¿Cuál es el nivel de vuelo de XYZ456?", False),
    ("Consulta el plan de vuelo de NOEXISTE", False),
    ("Selecciona la aeronave y comprueba su panel", True),
    ("Selecciona ABC123", False),
    ("¿Qué diferencia hay entre un plan y el panel en este simulador?", False),
]
# question, allowed = examples[0]
# show_task(run_task(get_client(), question, ATMSimulator(), allow_selection=allowed))

## 11. ¿Qué tiene esto que ver con Robot Framework y MCP?

La entrada que queremos automatizar puede ser un paso de prueba escrito en lenguaje natural. Las herramientas proporcionan operaciones concretas con contratos conocidos. El LLM interpreta la intención y propone una secuencia. Python ejecuta y verifica.

Una futura variante puede producir un plan estructurado sin ejecutarlo, validarlo y transformarlo en un fichero `.robot` con keywords existentes. Otra variante puede ejecutar las herramientas directamente bajo control. Son dos productos diferentes; ambos necesitan validación de argumentos, estados y resultados esperados.

En este notebook `TOOLS` es un catálogo local y `dispatch()` llama funciones Python. En el siguiente paso, un cliente MCP descubrirá las herramientas ofrecidas por un servidor y el dispatcher podrá invocarlas por el protocolo MCP. MCP no es un LLM ni reemplaza la orquestación o los permisos.

RAG es opcional para este caso: lo necesitaremos cuando haga falta recuperar procedimientos, requisitos o ejemplos para interpretar correctamente las pruebas. No es necesario para invocar una herramienta cuyo contrato ya conocemos.

## 12. Ejercicio y cierre

- Cambia el texto de prueba sin cambiar su significado: comprueba si el modelo mantiene la secuencia lógica.
- Añade una herramienta de solo lectura, por ejemplo `get_selected_aircraft`, y registra su esquema y su implementación en el dispatcher. ¿Qué sucede si registras solo el esquema?
- Experimenta con una descripción ambigua de dos herramientas. ¿Cómo afecta a la elección?
- Introduce otro defecto en el panel. Comprueba si el veredicto Python detecta la diferencia.

Preguntas clave:

1. ¿Quién ejecuta realmente la función?
2. ¿Qué información recibe el LLM sobre cada herramienta?
3. ¿Qué garantiza `strict` y qué no garantiza?
4. ¿Por qué necesitamos `call_id` y un bucle?
5. ¿Qué diferencia hay entre una respuesta textual, una solicitud de llamada y un resultado ejecutado?
6. ¿Por qué el veredicto PASS/FAIL se calcula fuera del LLM?
7. ¿Qué cambia al sustituir las funciones locales por un servidor MCP?

Siguiente notebook: `06_mcp_atm.ipynb`. Descubrimiento e invocación de herramientas mediante MCP, conservando la separación entre modelo, cliente, servidor y verificador.